# 音乐理解与智能混音助手 · Colab 特征提取流水线

**这个 notebook 只做一件事：下载 MTG-Jamendo 全量音频并提 MERT 特征。**
训练不在这里跑 —— 标签头只有 42 万参数，一次训练 96 秒，
本机跑比往返传数据快得多。Colab 的价值在于**数据中心带宽**和 GPU 前向。

### 为什么必须逐块流式处理

| | 体积 |
|---|---|
| 全量音频（100 块） | **~490 GB** |
| Colab 本地盘 | ~100–200 GB |
| 提出来的 4 段特征 | ~140 GB |

音频**塞不进本地盘**，所以流程必须是：
`下载一块 → 提特征 → 打包特征到云盘 → 删掉这块音频 → 下一块`。

### 为什么特征要打包成 tar 再传云盘

全量特征是 **5 万多个小 `.npy` 文件**。Google Drive 对大量小文件的写入极慢，
而且容易触发 API 配额。**每块打成一个 tar** → 100 个文件，稳定得多。

> ⚠️ **许可**：MTG-Jamendo 为 **非商业研究/学术用途**，音频与元数据**不可再分发**。
> 本 notebook 把数据下到你自己的云盘供你自己研究使用；**不要**把音频或特征公开分享。


## 0 · 挂载云盘并确认目录


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, shutil, subprocess, time

# 你在云盘里建好的目录（注意 'Audio AI' 中间有空格，所有路径都要加引号）
DRIVE = pathlib.Path('/content/drive/MyDrive/Audio AI/MusicMixer')
DRIVE.mkdir(parents=True, exist_ok=True)

# 特征包与元数据存云盘；音频只存本地盘（用完即删，永远不进云盘）
FEAT_DRIVE = DRIVE / 'features_tar'   ; FEAT_DRIVE.mkdir(exist_ok=True)
META_DRIVE = DRIVE / 'meta'           ; META_DRIVE.mkdir(exist_ok=True)
LOCAL      = pathlib.Path('/content/work')

free = shutil.disk_usage('/content').free / 2**30
print(f'云盘目录  : {DRIVE}')
print(f'本地可用盘: {free:.0f} GB')
assert free > 30, '本地盘不足 30 GB，无法容纳一块音频 + 特征'


## 1 · 拉取项目代码

**直接 clone 仓库，不在这里重写任何逻辑。**
notebook 里复制一份提特征代码看似方便，但两边会各自演化，
最后 Colab 出的特征和本机出的对不上 —— 而这种不一致是静默的。


In [ ]:
REPO = 'https://github.com/EthanBAI-dev/musicmix.git'
SRC  = pathlib.Path('/content/musicmix')

if SRC.exists():
    !cd {SRC} && git pull --ff-only
else:
    !git clone --depth 1 {REPO} {SRC}

%cd {SRC}
!git log --oneline -1


## 2 · 装依赖


In [ ]:
!pip -q install librosa soundfile pyloudnorm transformers torchaudio nnAudio

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), '没拿到 GPU：菜单 代码执行程序 → 更改运行时类型 → GPU'


## 3 · 配置

`SEGMENTS` 要和本机实验保持一致，否则数字不可比。
当前本机最佳配置是 **第 6 层 · 4 段**（P5 结论：4 段比中间 30 秒 +0.0268 mAP，5/5 种子一致）。


In [ ]:
LAYER      = 6
SEGMENTS   = 4          # 与本机一致；改这个数会让结果无法与既有实验比较
CHUNK_FROM = 10         # 本机已有 0–9，从 10 开始接着下
CHUNK_TO   = 100        # 全量共 100 块
SUBSET     = 'autotagging_top50tags'

DATA = SRC / 'data' / 'jamendo'
DATA.mkdir(parents=True, exist_ok=True)

# 元数据只需一次，且很小，直接放云盘并软链回来（重启会话不用重下）
if not (META_DRIVE / 'autotagging.tsv').exists():
    !python -m scripts.download_jamendo --meta-only --root {DATA}
    !cp -n {DATA}/meta/*.tsv {DATA}/meta/*.txt '{META_DRIVE}/' 2>/dev/null || true
else:
    (DATA / 'meta').mkdir(exist_ok=True)
    !cp -n '{META_DRIVE}'/* {DATA}/meta/ 2>/dev/null || true
print('元数据文件:', len(list((DATA/'meta').glob('*'))))


## 4 · 主循环：下载 → 提特征 → 打包 → 删音频

**可以随时中断重跑。** 云盘上已存在 tar 的块会自动跳过，
所以 Colab 断开会话（12 小时上限）之后重新执行这一格即可接着走。


In [ ]:
# 特征目录名不要手写 —— 直接问代码要。
# 手写一份格式串意味着仓库里改了命名规则，这个 notebook 会**静默**指向错误目录：
# 提特征成功、打包出空 tar、校验才发现，白跑几小时。
import sys; sys.path.insert(0, str(SRC))
from src.tagging.backbone import BackboneConfig, MERT_95M

CFG = BackboneConfig(name=MERT_95M, layer=LAYER, n_segments=SEGMENTS)
FEAT_DIR = DATA / 'features' / CFG.tag()
print('特征目录:', FEAT_DIR.name)

def tar_path(i):  return FEAT_DRIVE / f'chunk{i:02d}_{CFG.tag()}.tar'
def audio_dir(i): return DATA / 'audio' / f'{i:02d}'

t_start = time.time()
for i in range(CHUNK_FROM, CHUNK_TO):
    if tar_path(i).exists():
        print(f'[{i:02d}] 云盘已有特征包，跳过'); continue
    t0 = time.time()

    # (1) 只下这一块
    !python -m scripts.download_jamendo --start {i} --chunks {i+1} --root {DATA}
    if not audio_dir(i).exists():
        print(f'[{i:02d}] ⚠️ 下载失败，跳过（重跑本格会自动重试）'); continue

    # (2) 提特征。已缓存的会跳过，所以实际只算这一块的新曲目
    !python -m scripts.extract_backbone --layer {LAYER} --segments {SEGMENTS} --subset {SUBSET} --root {DATA} --device cuda

    # (3) 打包到云盘。先在本地盘打包再整体搬 —— 直接往 Drive 里边写边传很容易半途出错
    sub = FEAT_DIR / f'{i:02d}'
    if sub.exists():
        tmp = pathlib.Path(f'/content/chunk{i:02d}.tar')
        !tar -cf {tmp} -C {FEAT_DIR} {i:02d}
        shutil.move(str(tmp), tar_path(i))
    else:
        print(f'[{i:02d}] ⚠️ 没有产出特征目录 {sub}，跳过打包')

    # (4) 删音频给下一块腾地方 —— 这是整个流程能跑通的关键
    shutil.rmtree(audio_dir(i), ignore_errors=True)

    el, tot = time.time()-t0, time.time()-t_start
    left = CHUNK_TO - i - 1
    print(f'[{i:02d}] ✅ {el/60:.1f} min | 累计 {tot/3600:.1f} h | '
          f'剩 {left} 块，预计还要 {left*el/3600:.1f} h', flush=True)

print('\n🎉 全部完成')


## 5 · 校验：特征数量对不对得上元数据

**别只看「没报错」。** 本项目吃过亏：一次 tar 截断只解出 191/556 个文件，
而脚本只检查了「目录非空」，于是重跑永远跳过那一块。
这里逐块比对**实际文件数**与元数据里该块应有的曲目数。


In [ ]:
import tarfile, collections

# 元数据里每块应有多少首（只算 top50tags 子集）
want = collections.Counter()
with open(DATA/'meta'/f'{SUBSET}.tsv', encoding='utf-8') as f:
    next(f)
    for line in f:
        for c in line.rstrip('\n').split('\t'):
            if c.endswith('.mp3'):
                want[c.split('/')[0]] += 1
                break

bad = []
for i in range(CHUNK_FROM, CHUNK_TO):
    p = tar_path(i)
    if not p.exists():
        bad.append((f'{i:02d}', 'tar 缺失', 0, want[f'{i:02d}'])); continue
    with tarfile.open(p) as t:
        n = sum(1 for m in t.getmembers() if m.name.endswith('.npy'))
    w = want[f'{i:02d}']
    # 允许 2% 缺口：少数曲目解码失败是正常的，成片缺失才是问题
    if w and n < w * 0.98:
        bad.append((f'{i:02d}', '数量不足', n, w))

if bad:
    print('❌ 以下块有问题，删掉对应 tar 后重跑第 4 格：')
    for c, why, n, w in bad: print(f'   chunk {c}  {why}  {n}/{w}')
else:
    print(f'✅ chunk {CHUNK_FROM}–{CHUNK_TO-1} 全部通过校验')


## 6 · 取回本机

在**本机**终端跑（不是在 Colab 里）：

```bash
rclone copy gdrive:'Audio AI/MusicMixer/features_tar' ./tars --progress
```

或直接从 Drive 网页下载。然后解包到特征目录：

```bash
for f in tars/*.tar; do tar -xf "$f" -C data/jamendo/features/MERT-v1-95M_L6_s5_30s_x4/; done
```

**特征总量约 140 GB**，按需只取要用的块即可 —— 
先用一半验证结论是否随数据量变化，比一次拉满更稳妥。

### 之后在本机训练

```bash
python -m scripts.train_tagging --level L2 --arch attnhead --pooling mean \
    --layer 6 --segments 4 --seed 0 --out results/seeds/FULL_s0.json
```

**96 秒一次。** 不要为了这一步再回 Colab。
